In [3]:
import pandas as pd
import numpy as np

In [53]:
data = pd.read_csv('data/data.csv',
                dtype={28: "string", 66: "string"}) # NOTE: Simply running read_csv('csv') produced Dtypewarning: Columns (28, 66) have mixed types. Perhaps we should look into what's going on there?
print("Done")

Done


In [54]:
print(f"Size: {data.shape[0]} rows by {data.shape[1]} columns")
data.columns = data.columns.str.lower()

Size: 292658 rows by 87 columns


### Empty Columns

This is a messy dataset so uh, yeah gl y'all! I believe in you 

Current progress: I've been considering just the columns with >= 20% null values. It is still worth looking into why any other columns even have null values.

In [55]:
summary = (
    pd.DataFrame({
        "percent_null": data.isna().mean() * 100,
        "null_count": data.isna().sum(),
        "dtype": data.dtypes.astype(str),
    })
    .query("percent_null > 20")
    .sort_values("percent_null", ascending=False)
    .reset_index()
)

flagged_cols = summary['index'].tolist()
hundreds_cols = summary[:13]['index'].tolist()
eighty_ninety_cols = summary[14:31]['index'].tolist()
fifty_seventy_cols = summary[32:44]['index'].tolist()
print(summary.round(2))

                                  index  percent_null  null_count    dtype
0                names.default.fallback        100.00      292658  float64
1                       names.mro.value        100.00      292658  float64
2         descriptions.default.fallback        100.00      292658  float64
3    shortdescriptions.default.fallback        100.00      292658  float64
4                         variantfields        100.00      292658  float64
5                     availability_date        100.00      292658  float64
6           metatitles.default.fallback        100.00      292658  float64
7              slugprototypes.mro.value        100.00      292658  float64
8       slugprototypes.default.fallback        100.00      292658  float64
9           metakeywords.livhaven.value        100.00      292658  float64
10    metadescriptions.default.fallback        100.00      292658  float64
11        metakeywords.default.fallback        100.00      292658  float64
12               metakeyw

In [34]:
data.drop(columns=hundreds_cols, inplace=True)

Here we see 14 columns that are completely null, so for the purposes of this research we will remove them for now.
The other columns prompt more investigation, since it's not immediately clear why so many of its values are missing. Next we work on the columns with percentages at the 80's-90's.

In [56]:
for column in eighty_ninety_cols:
    cleanedCol = data[column].dropna()
    print(f"total length: {len(cleanedCol)}\n{cleanedCol.head(5)}\n")

total length: 893
135    Bosch Rexroth, Aluminum Framing, Extrusions, p...
228    Bosch Rexroth, Aluminum Framing, Extrusions, p...
334    Bosch Rexroth, Aluminum Framing, Extrusions, p...
335    Bosch Rexroth, Aluminum Framing, Extrusions, p...
336    Bosch Rexroth, Aluminum Framing, Extrusions, p...
Name: metakeywords.default.value, dtype: string

total length: 1046
70     category
89     category
118    category
144    category
202    category
Name: decrementquantity.value, dtype: object

total length: 1539
70     category
89     category
118    category
144    category
202    category
Name: backorder.value, dtype: object

total length: 1539
70     category
89     category
118    category
144    category
202    category
Name: manageinventory.value, dtype: object

total length: 1539
70     category
89     category
118    category
144    category
202    category
Name: maximumquantitytoorder.value, dtype: object

total length: 1539
70     category
89     category
118    category
144   

Above are the sample outputs of the values between the 80-90% range. Some of it looks droppable, such as "custom_quantity_increment" 
since it looks manual, and similarly for those that say "category". Some of the columns look necessary for our work, such as the one with only 893 non-na entries saying "Bosch Rexroth, Aluminum Framing, Extrusions, p..." Perhaps we can combine these columns with another existing, meaningful column?